# 🎨 Professional Template Studio — v3 (Full Featured)

**সব cell top-to-bottom একবার run করুন, তারপর live edit করুন।**

### ✅ সব Features:
| Feature | Cell |
|---|---|
| Canvas + Aspect Ratio Presets (Story/Post/LinkedIn/YT) | Tab 1 |
| Top Accent Color Bar | Tab 2 |
| Background Image Upload + Blend | Tab 2 |
| Pattern Overlay (dots/grid/diagonal/hexagon) | Tab 2 |
| Header + Hero 3 Lines + Font Selector | Tab 3 |
| Module Box + Badge Shape (circle/square/diamond) + Drop Shadow | Tab 4 |
| Content + Stats/Number Block | Tab 5 |
| Divider Style (solid/dotted/double/gradient) | Tab 6 |
| Footer + Logo/Watermark Upload | Tab 7 |
| PNG/JPEG/WebP Export | Tab 8 |
| 12 Color Themes | Preset buttons |
| Snapshot + Undo (5 levels) | Action bar |
| Save/Load Settings JSON | Action bar |


In [ ]:
# @title 🚀 Step 1: Initialize System
import subprocess, sys, os, io, textwrap, time, json as _json, copy, math
from collections import deque
from IPython.display import display, clear_output, HTML

packages = {"ipywidgets": "ipywidgets", "pillow": "PIL", "requests": "requests"}
for pkg, imp in packages.items():
    try:
        __import__(imp)
    except ImportError:
        print(f"⏳ Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import ipywidgets as widgets
from PIL import Image, ImageDraw, ImageFont, ImageFilter

print("✅ System Ready!")


In [ ]:
# @title 🔤 Step 2: Font Management
import requests

class FontManager:
    _fonts = {}
    _loaded = False

    @classmethod
    def load_fonts(cls):
        if cls._loaded:
            return cls._fonts
        urls = {
            "Bold":    "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Bold.ttf",
            "Medium":  "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Medium.ttf",
            "Regular": "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
            "NotoB":   "https://github.com/google/fonts/raw/main/ofl/notosansbengali/NotoSansBengali%5Bwdth%2Cwght%5D.ttf",
        }
        os.makedirs("fonts", exist_ok=True)
        for name, url in urls.items():
            path = f"fonts/Font_{name}.ttf"
            if not os.path.exists(path):
                print(f"  ⏳ Downloading {name}...")
                r = requests.get(url, timeout=15)
                with open(path, "wb") as f:
                    f.write(r.content)
            cls._fonts[name] = path
        cls._loaded = True
        return cls._fonts

print("🔤 Loading fonts...")
FONT_PATHS = FontManager.load_fonts()
PRIMARY_FONT   = FONT_PATHS["Bold"]
SECONDARY_FONT = FONT_PATHS["Medium"]
REGULAR_FONT   = FONT_PATHS["Regular"]
FONT_MAP = {"Regular": REGULAR_FONT, "Medium": SECONDARY_FONT, "Bold": PRIMARY_FONT}
print("✅ All fonts loaded!")


In [ ]:
# @title 🛠️ Step 3: Graphic Utilities

def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def create_gradient(width, height, start_hex, end_hex, direction="vertical"):
    img  = Image.new("RGB", (width, height))
    draw = ImageDraw.Draw(img)
    s, e = hex_to_rgb(start_hex), hex_to_rgb(end_hex)
    if direction == "vertical":
        for i in range(height):
            t = i / max(height-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(0,i),(width,i)], fill=c)
    elif direction == "horizontal":
        for i in range(width):
            t = i / max(width-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(i,0),(i,height)], fill=c)
    elif direction == "diagonal":
        steps = width + height
        for i in range(steps):
            t = i / max(steps-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(max(0,i-height), min(i,height)),
                       (min(i,width),   max(0,i-width))], fill=c)
    return img

def draw_text_with_shadow(draw, text, font, fill, x, y,
                           shadow_color="#000000", offset=(3,3)):
    draw.text((x+offset[0], y+offset[1]), text, font=font, fill=shadow_color)
    draw.text((x, y), text, font=font, fill=fill)

def get_text_x(text, font, align="left", margin_x=80, canvas_w=1080):
    try:    tw = font.getlength(text)
    except: tw = len(text) * max(font.size // 2, 1)
    if align == "center": return int((canvas_w - tw) // 2)
    if align == "right":  return int(canvas_w - margin_x - tw)
    return margin_x

def draw_text_wrapped(draw, text, font, color, x, y, max_width, line_spacing=10):
    try:    cw = max(font.getlength("A"), 1)
    except: cw = max(font.size // 2, 1)
    chars = max(1, int(max_width / cw))
    lines = textwrap.wrap(text, width=chars) or [text]
    cur_y = y
    for line in lines:
        draw.text((x, cur_y), line, font=font, fill=color)
        bbox  = draw.textbbox((0,0), line, font=font)
        cur_y += (bbox[3]-bbox[1]) + line_spacing
    return cur_y

def draw_accent_box(draw, x1, y1, x2, y2, radius, bg_color,
                    border_color=None, border_width=3):
    draw.rounded_rectangle([(x1,y1),(x2,y2)], radius=radius, fill=bg_color)
    if border_color:
        draw.rounded_rectangle([(x1,y1),(x2,y2)],
                                radius=radius, outline=border_color, width=border_width)

# ── NEW: Pattern Overlay ──────────────────────────────────────────
def apply_pattern_overlay(base_img, pattern_type, hex_color, opacity, spacing=40):
    w, h   = base_img.size
    overlay = Image.new("RGBA", (w, h), (0,0,0,0))
    drw     = ImageDraw.Draw(overlay)
    r,g,b   = hex_to_rgb(hex_color)
    alpha   = int(255 * min(max(opacity, 0), 1))
    c       = (r, g, b, alpha)
    sp      = max(spacing, 5)

    if pattern_type == "dots":
        rad = max(2, sp // 10)
        for y in range(0, h+sp, sp):
            for x in range(0, w+sp, sp):
                drw.ellipse([(x-rad,y-rad),(x+rad,y+rad)], fill=c)
    elif pattern_type == "grid":
        for y in range(0, h, sp):
            drw.line([(0,y),(w,y)], fill=c, width=1)
        for x in range(0, w, sp):
            drw.line([(x,0),(x,h)], fill=c, width=1)
    elif pattern_type == "diagonal":
        for i in range(-(h+sp), w+h+sp, sp):
            drw.line([(i,0),(i+h+sp,h+sp)], fill=c, width=1)
    elif pattern_type == "hexagon":
        hw = sp
        hh = max(int(sp * 0.866), 1)
        for row in range(-1, h//hh+2):
            for col in range(-1, w//hw+2):
                ox  = (hw//2) if row%2 else 0
                cx2 = col*hw + ox
                cy2 = row*hh
                pts = [(cx2+int((hw//2)*math.cos(math.radians(a))),
                        cy2+int((hw//2)*math.sin(math.radians(a))))
                       for a in range(0,360,60)]
                drw.polygon(pts, outline=c)

    base_rgba = base_img.convert("RGBA")
    return Image.alpha_composite(base_rgba, overlay).convert("RGB")

# ── NEW: BG Image Blend ───────────────────────────────────────────
def apply_bg_image(base_img, pil_img, opacity):
    if pil_img is None:
        return base_img
    w, h = base_img.size
    bg   = pil_img.resize((w,h), Image.LANCZOS).convert("RGBA")
    r,g,b,a = bg.split()
    a = a.point(lambda p: int(p * min(max(opacity,0),1)))
    bg.putalpha(a)
    base_rgba = base_img.convert("RGBA")
    return Image.alpha_composite(base_rgba, bg).convert("RGB")

# ── NEW: Logo Placement ───────────────────────────────────────────
def apply_logo(base_img, logo_pil, size, position, opacity):
    if logo_pil is None:
        return base_img
    w, h   = base_img.size
    lw, lh = logo_pil.size
    ratio  = size / max(lw, lh, 1)
    ns     = (max(1,int(lw*ratio)), max(1,int(lh*ratio)))
    logo   = logo_pil.resize(ns, Image.LANCZOS).convert("RGBA")
    r,g,b,a = logo.split()
    a = a.point(lambda p: int(p * min(max(opacity,0),1)))
    logo.putalpha(a)
    lw2, lh2 = logo.size
    pad = 30
    pos_map = {
        "top-left":     (pad, pad),
        "top-right":    (w-lw2-pad, pad),
        "bottom-left":  (pad, h-lh2-pad),
        "bottom-right": (w-lw2-pad, h-lh2-pad),
    }
    pos    = pos_map.get(position, (pad, pad))
    base_rgba = base_img.convert("RGBA")
    base_rgba.paste(logo, pos, logo)
    return base_rgba.convert("RGB")

# ── NEW: Module Box Shadow ────────────────────────────────────────
def apply_box_shadow(img, x1, y1, x2, y2, radius, offset=12, blur=10):
    w, h = img.size
    shadow = Image.new("RGBA", (w,h), (0,0,0,0))
    sd     = ImageDraw.Draw(shadow)
    sd.rounded_rectangle(
        [(x1+offset//2, y1+offset), (x2+offset//2, y2+offset)],
        radius=radius, fill=(0,0,0,110)
    )
    shadow = shadow.filter(ImageFilter.GaussianBlur(blur))
    base   = img.convert("RGBA")
    return Image.alpha_composite(base, shadow).convert("RGB")

# ── NEW: Badge Shape ──────────────────────────────────────────────
def draw_badge_shape(draw, cx, cy, radius, shape, fill_color,
                     border_color=None, border_width=2):
    if shape == "circle":
        draw.ellipse([(cx-radius,cy-radius),(cx+radius,cy+radius)],
                     fill=fill_color, outline=border_color,
                     width=border_width if border_color else 0)
    elif shape == "square":
        r = int(radius*0.85)
        draw.rounded_rectangle([(cx-r,cy-r),(cx+r,cy+r)], radius=10,
                                fill=fill_color, outline=border_color,
                                width=border_width if border_color else 0)
    elif shape == "diamond":
        pts = [(cx, cy-radius),(cx+radius, cy),(cx, cy+radius),(cx-radius, cy)]
        draw.polygon(pts, fill=fill_color, outline=border_color)

# ── NEW: Advanced Divider ─────────────────────────────────────────
def draw_divider_adv(img, x1, x2, y, color, style="solid",
                     thickness=2, color2=None):
    """Modifies img in-place; returns img for chaining."""
    draw = ImageDraw.Draw(img)
    if style == "solid":
        draw.line([(x1,y),(x2,y)], fill=color, width=thickness)
    elif style == "dotted":
        dash, x = 15, x1
        while x < x2:
            draw.line([(x,y),(min(x+dash,x2),y)], fill=color, width=thickness)
            x += dash*2
    elif style == "double":
        draw.line([(x1,y),(x2,y)], fill=color, width=1)
        draw.line([(x1,y+4),(x2,y+4)], fill=color, width=1)
    elif style == "gradient":
        s  = hex_to_rgb(color)
        e  = hex_to_rgb(color2 or "#000000")
        st = x2 - x1
        for i in range(st):
            t = i / max(st-1, 1)
            c = tuple(int(s[j]+(e[j]-s[j])*t) for j in range(3))
            draw.line([(x1+i,y),(x1+i,y+thickness-1)], fill=c)
    return img

# ── NEW: Get uploaded PIL image ───────────────────────────────────
def get_uploaded_image(upload_widget):
    if not upload_widget.value:
        return None
    try:
        val = upload_widget.value
        content = list(val.values())[0]["content"] if isinstance(val, dict) else val[0]["content"]
        return Image.open(io.BytesIO(content)).convert("RGBA")
    except Exception:
        return None

print("✅ Utilities Ready!")


In [ ]:
# @title 🎨 Step 4: Rendering Engine

def render_template(
    # Canvas
    cv_w, cv_h, bg_start, bg_end, grad_dir,
    # Accent Bar
    show_accent_bar, accent_bar_h, accent_bar_col,
    # BG Image
    bg_img, bg_img_opacity,
    # Pattern
    show_pattern, pattern_type, pattern_col, pattern_opacity, pattern_spacing,
    # Header
    head_txt, head_sz, head_col, head_y, head_align, head_font,
    # Hero 1 & 2
    hero1_txt, hero1_sz, hero1_col, hero1_y,
    hero2_txt, hero2_sz, hero2_col, hero2_y,
    # Hero 3
    show_hero3, hero3_txt, hero3_sz, hero3_col, hero3_y,
    # Hero misc
    hero_shadow, hero_align, hero_font,
    # Module Box
    show_mod, mod_y, mod_h, mod_bg, mod_rad,
    show_mod_border, mod_border_col, mod_shadow,
    # Badge
    bdg_num, bdg_title, bdg_sz, bdg_col, bdg_shape,
    # Content
    err_title, err_txt, err_col,
    opt_title, opt_txt, opt_col,
    body_font,
    # Stats Block
    show_stats, stats_y,
    stats_num1, stats_lbl1,
    stats_num2, stats_lbl2,
    stats_num3, stats_lbl3,
    stats_col,
    # Logo
    logo_img, logo_size, logo_pos, logo_opacity, show_logo,
    # Footer
    show_footer, foot_txt, foot_col, foot_sz,
    # Layout
    margin_x, spacing,
    # Divider
    show_divider, divider_col, divider_style,
):
    try:
        t0 = time.time()

        # Fonts
        f_head    = ImageFont.truetype(FONT_MAP.get(head_font, REGULAR_FONT), head_sz)
        f_h1      = ImageFont.truetype(FONT_MAP.get(hero_font, PRIMARY_FONT), hero1_sz)
        f_h2      = ImageFont.truetype(FONT_MAP.get(hero_font, PRIMARY_FONT), hero2_sz)
        f_h3      = ImageFont.truetype(FONT_MAP.get(hero_font, PRIMARY_FONT), hero3_sz)
        f_bdgn    = ImageFont.truetype(PRIMARY_FONT, bdg_sz)
        f_bdgt    = ImageFont.truetype(PRIMARY_FONT, 30)
        f_bt      = ImageFont.truetype(FONT_MAP.get(body_font, PRIMARY_FONT), 24)
        f_btxt    = ImageFont.truetype(FONT_MAP.get(body_font, SECONDARY_FONT), 28)
        f_foot    = ImageFont.truetype(REGULAR_FONT, foot_sz)
        f_stn     = ImageFont.truetype(PRIMARY_FONT, 64)
        f_stl     = ImageFont.truetype(SECONDARY_FONT, 22)

        # ── 1. Canvas ─────────────────────────────────────────────
        img  = create_gradient(cv_w, cv_h, bg_start, bg_end, direction=grad_dir)

        # ── 2. BG Image ───────────────────────────────────────────
        if bg_img is not None:
            img = apply_bg_image(img, bg_img, bg_img_opacity)

        # ── 3. Pattern Overlay ────────────────────────────────────
        if show_pattern:
            img = apply_pattern_overlay(img, pattern_type, pattern_col,
                                         pattern_opacity, pattern_spacing)

        draw = ImageDraw.Draw(img)

        # ── 4. Top Accent Bar ─────────────────────────────────────
        if show_accent_bar and accent_bar_h > 0:
            draw.rectangle([(0,0),(cv_w, accent_bar_h)], fill=accent_bar_col)

        # ── 5. Header ─────────────────────────────────────────────
        hx = get_text_x(head_txt, f_head, head_align, margin_x, cv_w)
        draw.text((hx, head_y), head_txt, font=f_head, fill=head_col)

        # ── 6. Hero Texts ─────────────────────────────────────────
        for txt, fnt, col, y_pos in [
            (hero1_txt, f_h1, hero1_col, hero1_y),
            (hero2_txt, f_h2, hero2_col, hero2_y),
        ]:
            tx = get_text_x(txt, fnt, hero_align, margin_x, cv_w)
            if hero_shadow:
                draw_text_with_shadow(draw, txt, fnt, col, tx, y_pos,
                                       shadow_color="#000033", offset=(4,4))
            else:
                draw.text((tx, y_pos), txt, font=fnt, fill=col)

        if show_hero3 and hero3_txt.strip():
            h3x = get_text_x(hero3_txt, f_h3, hero_align, margin_x, cv_w)
            if hero_shadow:
                draw_text_with_shadow(draw, hero3_txt, f_h3, hero3_col, h3x, hero3_y,
                                       shadow_color="#000033", offset=(4,4))
            else:
                draw.text((h3x, hero3_y), hero3_txt, font=f_h3, fill=hero3_col)

        # ── 7. Divider below hero ─────────────────────────────────
        if show_divider:
            last_y = (hero3_y + hero3_sz if (show_hero3 and hero3_txt.strip())
                      else hero2_y + hero2_sz)
            div_y = last_y + 20
            img   = draw_divider_adv(img, margin_x, cv_w-margin_x, div_y,
                                      divider_col, style=divider_style,
                                      thickness=2, color2=bg_end)
            draw  = ImageDraw.Draw(img)

        # ── 8. Module Box ─────────────────────────────────────────
        if show_mod:
            x1, y1 = margin_x, mod_y
            x2, y2 = cv_w-margin_x, mod_y+mod_h

            if mod_shadow:
                img  = apply_box_shadow(img, x1, y1, x2, y2, mod_rad)
                draw = ImageDraw.Draw(img)

            border_c = mod_border_col if show_mod_border else None
            draw_accent_box(draw, x1, y1, x2, y2, mod_rad, mod_bg,
                             border_color=border_c, border_width=3)

            # Badge
            bx, by   = x1+40, y1+40
            bcx, bcy = bx+30, by+30
            draw_badge_shape(draw, bcx, bcy, 30, bdg_shape,
                             "#1E293B", border_color=bdg_col, border_width=2)
            draw.text((bx+8, by+8),   bdg_num,   font=f_bdgn, fill=bdg_col)
            draw.text((bx+80, by+15), bdg_title, font=f_bdgt, fill="white")

            # Error
            cur_y = by + 110
            draw.text((bx, cur_y), err_title, font=f_bt, fill=err_col)
            cur_y += 40
            cur_y  = draw_text_wrapped(draw, err_txt, f_btxt, "#B0BEC5",
                                        bx+20, cur_y, (x2-x1)-60)

            # Inner divider
            draw.line([(bx, cur_y+spacing//2),(x2-40, cur_y+spacing//2)],
                       fill="#2D3748", width=1)

            # Solution
            cur_y += spacing+20
            draw.text((bx, cur_y), opt_title, font=f_bt, fill=opt_col)
            cur_y += 40
            draw_text_wrapped(draw, opt_txt, f_btxt, "white",
                               bx+20, cur_y, (x2-x1)-60)

        # ── 9. Stats Block ────────────────────────────────────────
        if show_stats:
            items = [(stats_num1,stats_lbl1),(stats_num2,stats_lbl2),(stats_num3,stats_lbl3)]
            valid = [(n,l) for n,l in items if n.strip()]
            if valid:
                uw    = cv_w - 2*margin_x
                col_w = uw // len(valid)
                for i,(num,lbl) in enumerate(valid):
                    cx = margin_x + i*col_w + col_w//2
                    try:    nw = int(f_stn.getlength(num))
                    except: nw = len(num)*35
                    try:    lw2 = int(f_stl.getlength(lbl))
                    except: lw2 = len(lbl)*12
                    draw_text_with_shadow(draw, num, f_stn, stats_col,
                                          cx-nw//2, stats_y,
                                          shadow_color="#000000", offset=(3,3))
                    draw.text((cx-lw2//2, stats_y+78), lbl,
                               font=f_stl, fill="#B0BEC5")

        # ── 10. Footer ────────────────────────────────────────────
        if show_footer and foot_txt.strip():
            fy = cv_h - foot_sz - 40
            draw.line([(margin_x, fy-20),(cv_w-margin_x, fy-20)],
                       fill="#2D3748", width=1)
            fx = get_text_x(foot_txt, f_foot, "center", margin_x, cv_w)
            draw.text((fx, fy), foot_txt, font=f_foot, fill=foot_col)

        # ── 11. Logo / Watermark ──────────────────────────────────
        if show_logo and logo_img is not None:
            img = apply_logo(img, logo_img, logo_size, logo_pos, logo_opacity)

        ms = (time.time()-t0)*1000
        return img, ms

    except Exception as e:
        import traceback; traceback.print_exc()
        err_img = Image.new("RGB",(600,200),"#3B0000")
        ImageDraw.Draw(err_img).text((10,80), f"Render Error: {e}", fill="white")
        return err_img, 0

print("✅ Render Engine Ready!")


In [ ]:
# @title 🎛️ Step 5: UI Controls – Canvas, Background & Header

style       = {"description_width": "160px"}
layout_full = widgets.Layout(width="98%")
layout_half = widgets.Layout(width="49%")
layout_btn  = widgets.Layout(width="140px", height="35px")

# ══ 1. CANVAS ════════════════════════════════════════════════════
w_cv_w     = widgets.IntSlider(value=1080, min=800, max=2000, step=10,
                                description="প্রস্থ:", style=style, layout=layout_full)
w_cv_h     = widgets.IntSlider(value=1350, min=800, max=2000, step=10,
                                description="উচ্চতা:", style=style, layout=layout_full)
w_bg_start = widgets.ColorPicker(value="#050814", description="BG শুরু:", style=style)
w_bg_end   = widgets.ColorPicker(value="#1A1030", description="BG শেষ:", style=style)
w_grad_dir = widgets.ToggleButtons(
    options=["vertical","horizontal","diagonal"], value="vertical",
    description="Gradient:", style={"description_width":"160px","button_width":"100px"})

# Aspect Ratio Presets
ASPECT_PRESETS = {
    "📱 Story 9:16": (1080,1920), "⬛ Post 1:1": (1080,1080),
    "💼 LinkedIn":   (1200, 628), "🎬 YT Thumb": (1280, 720),
    "📌 Pinterest":  (1000,1500), "🐦 Twitter":  (1600, 900),
}
def _mk_aspect(w,h):
    def _h(b): w_cv_w.value=w; w_cv_h.value=h
    return _h
_aspect_btns = []
for name,(w,h) in ASPECT_PRESETS.items():
    b = widgets.Button(description=name, layout=layout_btn)
    b.on_click(_mk_aspect(w,h))
    _aspect_btns.append(b)

# ══ 2. TOP ACCENT BAR ════════════════════════════════════════════
w_show_accent_bar = widgets.Checkbox(value=True, description="Top Accent Bar দেখান", style=style)
w_accent_bar_h    = widgets.IntSlider(value=8, min=2, max=60, description="Bar উচ্চতা:", style=style)
w_accent_bar_col  = widgets.ColorPicker(value="#00E5FF", description="Bar রঙ:", style=style)

# ══ 3. BACKGROUND IMAGE ══════════════════════════════════════════
w_bg_img_upload  = widgets.FileUpload(accept="image/*", multiple=False,
                                       description="BG Image Upload:",
                                       layout=widgets.Layout(width="300px"))
w_bg_img_opacity = widgets.FloatSlider(value=0.40, min=0.0, max=1.0, step=0.05,
                                        description="BG Opacity:", style=style)

# ══ 4. BACKGROUND PATTERN ════════════════════════════════════════
w_show_pattern    = widgets.Checkbox(value=False, description="Pattern দেখান", style=style)
w_pattern_type    = widgets.ToggleButtons(
    options=["dots","grid","diagonal","hexagon"], value="dots",
    description="Pattern:", style={"description_width":"80px","button_width":"90px"})
w_pattern_col     = widgets.ColorPicker(value="#FFFFFF", description="Pattern রঙ:", style=style)
w_pattern_opacity = widgets.FloatSlider(value=0.08, min=0.02, max=0.40, step=0.01,
                                         description="Opacity:", style=style)
w_pattern_spacing = widgets.IntSlider(value=40, min=15, max=120,
                                       description="Spacing:", style=style)

# ══ 5. HEADER ════════════════════════════════════════════════════
w_head_txt   = widgets.Text(value="SYSTEM PROTOCOL // ANALYSIS",
                             description="হেডার টেক্সট:", style=style, layout=layout_full)
w_head_sz    = widgets.IntSlider(value=20, min=10, max=60, description="সাইজ:", style=style)
w_head_col   = widgets.ColorPicker(value="#00E5FF", description="রঙ:", style=style)
w_head_y     = widgets.IntSlider(value=60, min=0, max=400, description="Y:", style=style)
w_head_align = widgets.ToggleButtons(options=["left","center","right"], value="left",
    description="Align:", style={"description_width":"80px","button_width":"80px"})
w_head_font  = widgets.Dropdown(options=["Regular","Medium","Bold"], value="Regular",
                                 description="Font Weight:", style=style)

# ══ 6. HERO TEXT (3 Lines) ═══════════════════════════════════════
w_hero1_txt = widgets.Text(value="কেন ছাত্রছাত্রীরা ভুল",
                            description="Hero লাইন ১:", style=style, layout=layout_full)
w_hero2_txt = widgets.Text(value="পদ্ধতিতে পড়াশোনা করে?",
                            description="Hero লাইন ২:", style=style, layout=layout_full)
w_hero3_txt = widgets.Text(value="",
                            description="Hero লাইন ৩:", style=style, layout=layout_full)
w_show_hero3 = widgets.Checkbox(value=False, description="লাইন ৩ দেখান", style=style)

w_hero1_sz  = widgets.IntSlider(value=90, min=30, max=160, description="লাইন ১ সাইজ:", style=style)
w_hero2_sz  = widgets.IntSlider(value=90, min=30, max=160, description="লাইন ২ সাইজ:", style=style)
w_hero3_sz  = widgets.IntSlider(value=70, min=30, max=160, description="লাইন ৩ সাইজ:", style=style)

w_hero1_col = widgets.ColorPicker(value="#FFFFFF", description="লাইন ১ রঙ:", style=style)
w_hero2_col = widgets.ColorPicker(value="#00E5FF", description="লাইন ২ রঙ:", style=style)
w_hero3_col = widgets.ColorPicker(value="#B0BEC5", description="লাইন ৩ রঙ:", style=style)

w_hero1_y   = widgets.IntSlider(value=150, min=50, max=700, description="লাইন ১ Y:", style=style)
w_hero2_y   = widgets.IntSlider(value=270, min=50, max=700, description="লাইন ২ Y:", style=style)
w_hero3_y   = widgets.IntSlider(value=390, min=50, max=800, description="লাইন ৩ Y:", style=style)

w_hero_shadow = widgets.Checkbox(value=True, description="Text Shadow চালু", style=style)
w_hero_align  = widgets.ToggleButtons(options=["left","center","right"], value="left",
    description="Align:", style={"description_width":"80px","button_width":"80px"})
w_hero_font   = widgets.Dropdown(options=["Regular","Medium","Bold"], value="Bold",
                                  description="Font Weight:", style=style)

print("✅ Canvas & Header Controls Ready!")


In [ ]:
# @title 🎛️ Step 6: UI Controls – Body, Stats, Logo & Export

# ══ 7. MODULE BOX ════════════════════════════════════════════════
w_show_mod       = widgets.Checkbox(value=True, description="মডিউল বক্স দেখান", style=style)
w_mod_y          = widgets.IntSlider(value=450, min=200, max=1200,
                                      description="বক্স Y:", style=style, layout=layout_full)
w_mod_h          = widgets.IntSlider(value=420, min=150, max=900,
                                      description="বক্স উচ্চতা:", style=style, layout=layout_full)
w_mod_bg         = widgets.ColorPicker(value="#121826", description="বক্স BG:", style=style)
w_mod_rad        = widgets.IntSlider(value=30, min=0, max=100,
                                      description="রাউন্ডনেস:", style=style)
w_show_mod_border= widgets.Checkbox(value=True, description="Accent Border দেখান", style=style)
w_mod_border_col = widgets.ColorPicker(value="#00E5FF", description="Border রঙ:", style=style)
w_mod_shadow     = widgets.Checkbox(value=True, description="Drop Shadow চালু", style=style)

# Badge
w_bdg_num   = widgets.Text(value="01", description="ব্যাজ নম্বর:", style=style)
w_bdg_title = widgets.Text(value="রিটেনশন ফ্যাক্টর",
                            description="মডিউল নাম:", style=style, layout=layout_full)
w_bdg_sz    = widgets.IntSlider(value=24, min=10, max=50, description="নম্বর সাইজ:", style=style)
w_bdg_col   = widgets.ColorPicker(value="#00E5FF", description="নম্বর রঙ:", style=style)
w_bdg_shape = widgets.ToggleButtons(options=["circle","square","diamond"], value="circle",
    description="Badge Shape:", style={"description_width":"120px","button_width":"90px"})

# ══ 8. CONTENT ═══════════════════════════════════════════════════
w_err_title = widgets.Text(value="DETECTED ERROR:", description="Error টাইটেল:", style=style)
w_err_txt   = widgets.Textarea(value="মনে মনে পড়া বা প্যাসিভ লার্নিং।",
                                description="Error বিস্তারিত:", rows=2,
                                style=style, layout=layout_full)
w_err_col   = widgets.ColorPicker(value="#FF5722", description="Error রঙ:", style=style)

w_opt_title = widgets.Text(value="OPTIMAL PATH:", description="Solution টাইটেল:", style=style)
w_opt_txt   = widgets.Textarea(value="অ্যাক্টিভ রিকল এবং স্পেসড রিপিটেশন।",
                                description="Solution বিস্তারিত:", rows=2,
                                style=style, layout=layout_full)
w_opt_col   = widgets.ColorPicker(value="#00C853", description="Solution রঙ:", style=style)
w_body_font = widgets.Dropdown(options=["Regular","Medium","Bold"], value="Medium",
                                description="Body Font:", style=style)

# ══ 9. STATS BLOCK ═══════════════════════════════════════════════
w_show_stats  = widgets.Checkbox(value=False, description="Stats Block দেখান", style=style)
w_stats_y     = widgets.IntSlider(value=1050, min=500, max=1800, description="Stats Y:", style=style, layout=layout_full)
w_stats_col   = widgets.ColorPicker(value="#00E5FF", description="সংখ্যার রঙ:", style=style)
w_stats_num1  = widgets.Text(value="৯৫%",    description="সংখ্যা ১:", style=style)
w_stats_lbl1  = widgets.Text(value="সাফল্যের হার",description="লেবেল ১:", style=style)
w_stats_num2  = widgets.Text(value="৪৮H",    description="সংখ্যা ২:", style=style)
w_stats_lbl2  = widgets.Text(value="কোর্স সময়",  description="লেবেল ২:", style=style)
w_stats_num3  = widgets.Text(value="১০K+",   description="সংখ্যা ৩:", style=style)
w_stats_lbl3  = widgets.Text(value="শিক্ষার্থী",  description="লেবেল ৩:", style=style)

# ══ 10. LOGO / WATERMARK ════════════════════════════════════════
w_show_logo     = widgets.Checkbox(value=False, description="Logo দেখান", style=style)
w_logo_upload   = widgets.FileUpload(accept="image/*", multiple=False,
                                      description="Logo Upload:",
                                      layout=widgets.Layout(width="300px"))
w_logo_size     = widgets.IntSlider(value=100, min=30, max=400,
                                     description="Logo সাইজ:", style=style)
w_logo_pos      = widgets.ToggleButtons(
    options=["top-left","top-right","bottom-left","bottom-right"],
    value="top-right", description="Logo অবস্থান:",
    style={"description_width":"120px","button_width":"110px"})
w_logo_opacity  = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05,
                                       description="Logo Opacity:", style=style)

# ══ 11. FOOTER ═══════════════════════════════════════════════════
w_show_footer = widgets.Checkbox(value=True, description="Footer দেখান", style=style)
w_foot_txt    = widgets.Text(value="© Your Name | your.website.com",
                              description="Footer টেক্সট:", style=style, layout=layout_full)
w_foot_col    = widgets.ColorPicker(value="#607D8B", description="Footer রঙ:", style=style)
w_foot_sz     = widgets.IntSlider(value=22, min=12, max=40, description="Footer সাইজ:", style=style)

# ══ 12. LAYOUT ═══════════════════════════════════════════════════
w_margin_x     = widgets.IntSlider(value=80, min=0, max=200,
                                    description="সাইড মার্জিন:", style=style)
w_spacing      = widgets.IntSlider(value=45, min=0, max=100,
                                    description="লাইন স্পেসিং:", style=style)
w_show_divider = widgets.Checkbox(value=True, description="Divider Line দেখান", style=style)
w_divider_col  = widgets.ColorPicker(value="#00E5FF", description="Divider রঙ:", style=style)
w_divider_style= widgets.ToggleButtons(
    options=["solid","dotted","double","gradient"], value="solid",
    description="Divider Style:",
    style={"description_width":"120px","button_width":"90px"})

# ══ 13. EXPORT ═══════════════════════════════════════════════════
w_export_fmt     = widgets.ToggleButtons(options=["PNG","JPEG","WEBP"], value="PNG",
    description="Format:", style={"description_width":"80px","button_width":"70px"})
w_export_quality = widgets.IntSlider(value=92, min=50, max=100,
                                      description="Quality:", style=style)
w_filename       = widgets.Text(value="my_template_design",
                                 description="File Name:", style=style, layout=layout_full)

print("✅ Body Controls Ready!")


In [ ]:
# @title 🎨 Step 7: Preset Color Themes (12 Professional Themes)

PRESETS = {
    "🌌 Dark Space":       {"bg_start":"#050814","bg_end":"#1A1030","head_col":"#00E5FF","hero1_col":"#FFFFFF","hero2_col":"#00E5FF","mod_bg":"#121826","mod_border_col":"#00E5FF","err_col":"#FF5722","opt_col":"#00C853","divider_col":"#00E5FF","foot_col":"#607D8B","accent_bar_col":"#00E5FF"},
    "🔥 Sunset Orange":    {"bg_start":"#1A0A00","bg_end":"#3D1200","head_col":"#FF6D00","hero1_col":"#FFFFFF","hero2_col":"#FF6D00","mod_bg":"#1E1000","mod_border_col":"#FF6D00","err_col":"#FF1744","opt_col":"#76FF03","divider_col":"#FF6D00","foot_col":"#795548","accent_bar_col":"#FF6D00"},
    "💜 Purple Haze":      {"bg_start":"#0D001A","bg_end":"#1A0030","head_col":"#EA80FC","hero1_col":"#FFFFFF","hero2_col":"#EA80FC","mod_bg":"#160020","mod_border_col":"#EA80FC","err_col":"#FF4081","opt_col":"#69F0AE","divider_col":"#EA80FC","foot_col":"#9E9E9E","accent_bar_col":"#EA80FC"},
    "🌿 Forest Green":     {"bg_start":"#001209","bg_end":"#002614","head_col":"#69F0AE","hero1_col":"#FFFFFF","hero2_col":"#69F0AE","mod_bg":"#001A0C","mod_border_col":"#69F0AE","err_col":"#FF5252","opt_col":"#CCFF90","divider_col":"#69F0AE","foot_col":"#546E7A","accent_bar_col":"#69F0AE"},
    "🌅 Golden Hour":      {"bg_start":"#1A1200","bg_end":"#2D1F00","head_col":"#FFD600","hero1_col":"#FFFFFF","hero2_col":"#FFD600","mod_bg":"#1A1300","mod_border_col":"#FFD600","err_col":"#FF6E40","opt_col":"#B9F6CA","divider_col":"#FFD600","foot_col":"#78909C","accent_bar_col":"#FFD600"},
    "🩺 Midnight Medical": {"bg_start":"#020B18","bg_end":"#061E36","head_col":"#40C4FF","hero1_col":"#E3F2FD","hero2_col":"#40C4FF","mod_bg":"#0A1929","mod_border_col":"#40C4FF","err_col":"#EF5350","opt_col":"#00E676","divider_col":"#1565C0","foot_col":"#546E7A","accent_bar_col":"#40C4FF"},
    "🖤 Carbon Black":     {"bg_start":"#0A0A0A","bg_end":"#1C1C1C","head_col":"#E0E0E0","hero1_col":"#FFFFFF","hero2_col":"#BDBDBD","mod_bg":"#141414","mod_border_col":"#424242","err_col":"#EF9A9A","opt_col":"#A5D6A7","divider_col":"#424242","foot_col":"#616161","accent_bar_col":"#757575"},
    "🌊 Deep Ocean":       {"bg_start":"#001428","bg_end":"#00264D","head_col":"#82B1FF","hero1_col":"#E8EAF6","hero2_col":"#82B1FF","mod_bg":"#001933","mod_border_col":"#448AFF","err_col":"#FF6090","opt_col":"#64FFDA","divider_col":"#1A237E","foot_col":"#5C6BC0","accent_bar_col":"#448AFF"},
    "🍒 Cherry Blossom":   {"bg_start":"#1A0010","bg_end":"#33001F","head_col":"#F48FB1","hero1_col":"#FCE4EC","hero2_col":"#F48FB1","mod_bg":"#200015","mod_border_col":"#F06292","err_col":"#FF5252","opt_col":"#80CBC4","divider_col":"#880E4F","foot_col":"#AD1457","accent_bar_col":"#F06292"},
    "🏆 Royal Gold":       {"bg_start":"#0D0900","bg_end":"#1F1400","head_col":"#FFC400","hero1_col":"#FFF8E1","hero2_col":"#FFD740","mod_bg":"#150D00","mod_border_col":"#FFC400","err_col":"#FF6F00","opt_col":"#CCFF90","divider_col":"#FF8F00","foot_col":"#8D6E63","accent_bar_col":"#FFC400"},
    "🧊 Arctic Frost":     {"bg_start":"#011627","bg_end":"#022B4A","head_col":"#B2EBF2","hero1_col":"#E0F7FA","hero2_col":"#80DEEA","mod_bg":"#01243E","mod_border_col":"#4DD0E1","err_col":"#FF8A65","opt_col":"#A5F3A5","divider_col":"#0097A7","foot_col":"#607D8B","accent_bar_col":"#4DD0E1"},
    "🔬 Neon Lab":         {"bg_start":"#030014","bg_end":"#0A0028","head_col":"#E040FB","hero1_col":"#EDE7F6","hero2_col":"#7C4DFF","mod_bg":"#07001E","mod_border_col":"#7C4DFF","err_col":"#FF4081","opt_col":"#69F0AE","divider_col":"#651FFF","foot_col":"#7E57C2","accent_bar_col":"#7C4DFF"},
}

_preset_widget_keys = ["bg_start","bg_end","head_col","hero1_col","hero2_col",
                        "mod_bg","mod_border_col","err_col","opt_col",
                        "divider_col","foot_col","accent_bar_col"]

preset_buttons = []
out_preset = widgets.Output()

def _mk_preset_handler(name, vals):
    def _h(b):
        _map = {
            "bg_start":      w_bg_start,
            "bg_end":        w_bg_end,
            "head_col":      w_head_col,
            "hero1_col":     w_hero1_col,
            "hero2_col":     w_hero2_col,
            "mod_bg":        w_mod_bg,
            "mod_border_col":w_mod_border_col,
            "err_col":       w_err_col,
            "opt_col":       w_opt_col,
            "divider_col":   w_divider_col,
            "foot_col":      w_foot_col,
            "accent_bar_col":w_accent_bar_col,
        }
        for k, wgt in _map.items():
            if k in vals:
                wgt.value = vals[k]
        with out_preset:
            clear_output()
            print(f"✅ Theme: {name}")
    return _h

for name, vals in PRESETS.items():
    btn = widgets.Button(description=name,
                          layout=widgets.Layout(width="195px", height="38px"),
                          style={"button_color":"#1E293B"})
    btn.on_click(_mk_preset_handler(name, vals))
    preset_buttons.append(btn)

_rows = [preset_buttons[i:i+4] for i in range(0, len(preset_buttons), 4)]
preset_box = widgets.VBox([
    widgets.HTML("<b style='font-size:14px;'>🎨 Color Themes (12)</b>"),
    *[widgets.HBox(row) for row in _rows],
    out_preset
])
print(f"✅ {len(PRESETS)} Themes Ready!")


In [ ]:
# @title 📑 Step 8: Layout – Tabs

# Tab 1 – Canvas
tab1 = widgets.VBox([
    widgets.HTML("<b>📐 Canvas Size & Gradient</b>"),
    w_cv_w, w_cv_h,
    widgets.HBox([w_bg_start, w_bg_end]),
    w_grad_dir,
    widgets.HTML("<hr><b>📱 Aspect Ratio Presets (এক ক্লিকে)</b>"),
    widgets.HBox(_aspect_btns[:3]),
    widgets.HBox(_aspect_btns[3:]),
])

# Tab 2 – Background Effects
tab2 = widgets.VBox([
    widgets.HTML("<b>🎨 Top Accent Bar</b>"),
    w_show_accent_bar,
    widgets.HBox([w_accent_bar_h, w_accent_bar_col]),
    widgets.HTML("<hr><b>🖼️ Background Image Upload</b>"),
    w_bg_img_upload, w_bg_img_opacity,
    widgets.HTML("<hr><b>🔲 Pattern Overlay</b>"),
    w_show_pattern, w_pattern_type,
    widgets.HBox([w_pattern_col, w_pattern_opacity]),
    w_pattern_spacing,
])

# Tab 3 – Header & Hero
tab3 = widgets.VBox([
    widgets.HTML("<b>📋 Header</b>"),
    w_head_txt,
    widgets.HBox([w_head_sz, w_head_col]),
    widgets.HBox([w_head_y, w_head_font]),
    w_head_align,
    widgets.HTML("<hr><b>🎯 Hero Text (৩ লাইন)</b>"),
    w_hero1_txt, w_hero2_txt,
    w_show_hero3, w_hero3_txt,
    widgets.HBox([w_hero1_sz, w_hero2_sz, w_hero3_sz]),
    widgets.HBox([w_hero1_col, w_hero2_col, w_hero3_col]),
    widgets.HBox([w_hero1_y, w_hero2_y, w_hero3_y]),
    w_hero_shadow, w_hero_align, w_hero_font,
])

# Tab 4 – Module Box & Badge
tab4 = widgets.VBox([
    widgets.HTML("<b>📦 Module Box</b>"),
    w_show_mod,
    widgets.HBox([w_mod_y, w_mod_h]),
    widgets.HBox([w_mod_bg, w_mod_rad]),
    widgets.HBox([w_show_mod_border, w_mod_border_col]),
    w_mod_shadow,
    widgets.HTML("<hr><b>🏷️ Badge</b>"),
    widgets.HBox([w_bdg_num, w_bdg_title]),
    widgets.HBox([w_bdg_sz, w_bdg_col]),
    w_bdg_shape,
])

# Tab 5 – Content & Stats
tab5 = widgets.VBox([
    widgets.HTML("<b>❌ Error Section</b>"),
    w_err_title, w_err_txt, w_err_col,
    widgets.HTML("<hr><b>✅ Solution Section</b>"),
    w_opt_title, w_opt_txt, w_opt_col,
    w_body_font,
    widgets.HTML("<hr><b>📊 Stats / Number Block</b>"),
    w_show_stats, w_stats_y, w_stats_col,
    widgets.HBox([w_stats_num1, w_stats_lbl1]),
    widgets.HBox([w_stats_num2, w_stats_lbl2]),
    widgets.HBox([w_stats_num3, w_stats_lbl3]),
])

# Tab 6 – Layout & Divider
tab6 = widgets.VBox([
    widgets.HTML("<b>📏 Spacing & Margin</b>"),
    w_margin_x, w_spacing,
    widgets.HTML("<hr><b>➖ Divider Line</b>"),
    w_show_divider,
    widgets.HBox([w_divider_col, w_divider_style]),
])

# Tab 7 – Footer & Logo
tab7 = widgets.VBox([
    widgets.HTML("<b>🔻 Footer</b>"),
    w_show_footer, w_foot_txt,
    widgets.HBox([w_foot_sz, w_foot_col]),
    widgets.HTML("<hr><b>🖼️ Logo / Watermark</b>"),
    w_show_logo, w_logo_upload,
    widgets.HBox([w_logo_size, w_logo_opacity]),
    w_logo_pos,
])

# Tab 8 – Export
tab8 = widgets.VBox([
    widgets.HTML("<b>📤 Export Options</b>"),
    w_export_fmt, w_export_quality, w_filename,
])

ui_tabs = widgets.Tab(children=[tab1,tab2,tab3,tab4,tab5,tab6,tab7,tab8])
for i,n in enumerate(["📐 Canvas","🎨 Background","✍️ Header & Hero",
                       "📦 Module","📝 Content & Stats",
                       "📏 Layout","🔻 Footer & Logo","📤 Export"]):
    ui_tabs.set_title(i, n)

print("✅ Tabs Ready!")


In [ ]:
# @title 🔄 Step 9: Live Engine + Undo + Save/Load

preview_output  = widgets.Output()
current_image   = None
_is_rendering   = False
_undo_stack     = deque(maxlen=5)   # Undo: last 5 snapshots

# Preview size slider
w_preview_size  = widgets.IntSlider(
    value=420, min=150, max=900, step=25,
    description="🔍 Preview সাইজ:",
    style={"description_width":"130px"},
    layout=widgets.Layout(width="500px"))

# ── widget_mapping (সব design widgets) ────────────────────────
widget_mapping = {
    # Canvas
    "cv_w":w_cv_w, "cv_h":w_cv_h, "bg_start":w_bg_start,
    "bg_end":w_bg_end, "grad_dir":w_grad_dir,
    # Accent Bar
    "show_accent_bar":w_show_accent_bar, "accent_bar_h":w_accent_bar_h,
    "accent_bar_col":w_accent_bar_col,
    # BG image opacity (upload widget handled separately)
    "bg_img_opacity":w_bg_img_opacity,
    # Pattern
    "show_pattern":w_show_pattern, "pattern_type":w_pattern_type,
    "pattern_col":w_pattern_col, "pattern_opacity":w_pattern_opacity,
    "pattern_spacing":w_pattern_spacing,
    # Header
    "head_txt":w_head_txt, "head_sz":w_head_sz, "head_col":w_head_col,
    "head_y":w_head_y, "head_align":w_head_align, "head_font":w_head_font,
    # Hero
    "hero1_txt":w_hero1_txt,"hero1_sz":w_hero1_sz,"hero1_col":w_hero1_col,"hero1_y":w_hero1_y,
    "hero2_txt":w_hero2_txt,"hero2_sz":w_hero2_sz,"hero2_col":w_hero2_col,"hero2_y":w_hero2_y,
    "show_hero3":w_show_hero3,
    "hero3_txt":w_hero3_txt,"hero3_sz":w_hero3_sz,"hero3_col":w_hero3_col,"hero3_y":w_hero3_y,
    "hero_shadow":w_hero_shadow,"hero_align":w_hero_align,"hero_font":w_hero_font,
    # Module
    "show_mod":w_show_mod,"mod_y":w_mod_y,"mod_h":w_mod_h,"mod_bg":w_mod_bg,
    "mod_rad":w_mod_rad,"show_mod_border":w_show_mod_border,
    "mod_border_col":w_mod_border_col,"mod_shadow":w_mod_shadow,
    # Badge
    "bdg_num":w_bdg_num,"bdg_title":w_bdg_title,"bdg_sz":w_bdg_sz,
    "bdg_col":w_bdg_col,"bdg_shape":w_bdg_shape,
    # Content
    "err_title":w_err_title,"err_txt":w_err_txt,"err_col":w_err_col,
    "opt_title":w_opt_title,"opt_txt":w_opt_txt,"opt_col":w_opt_col,
    "body_font":w_body_font,
    # Stats
    "show_stats":w_show_stats,"stats_y":w_stats_y,"stats_col":w_stats_col,
    "stats_num1":w_stats_num1,"stats_lbl1":w_stats_lbl1,
    "stats_num2":w_stats_num2,"stats_lbl2":w_stats_lbl2,
    "stats_num3":w_stats_num3,"stats_lbl3":w_stats_lbl3,
    # Logo
    "show_logo":w_show_logo,"logo_size":w_logo_size,
    "logo_pos":w_logo_pos,"logo_opacity":w_logo_opacity,
    # Footer
    "show_footer":w_show_footer,"foot_txt":w_foot_txt,
    "foot_col":w_foot_col,"foot_sz":w_foot_sz,
    # Layout
    "margin_x":w_margin_x,"spacing":w_spacing,
    "show_divider":w_show_divider,"divider_col":w_divider_col,
    "divider_style":w_divider_style,
    # Export
    "export_fmt":w_export_fmt,"export_quality":w_export_quality,
    "filename":w_filename,
}

def _collect_values():
    return {k: w.value for k, w in widget_mapping.items()}

def _do_render(vals):
    global current_image, _is_rendering
    if _is_rendering:
        return
    _is_rendering = True
    try:
        bg_pil   = get_uploaded_image(w_bg_img_upload)
        logo_pil = get_uploaded_image(w_logo_upload)

        with preview_output:
            clear_output(wait=True)
            img, ms = render_template(
                # Canvas
                vals["cv_w"], vals["cv_h"], vals["bg_start"], vals["bg_end"], vals["grad_dir"],
                # Accent Bar
                vals["show_accent_bar"], vals["accent_bar_h"], vals["accent_bar_col"],
                # BG Image
                bg_pil, vals["bg_img_opacity"],
                # Pattern
                vals["show_pattern"], vals["pattern_type"], vals["pattern_col"],
                vals["pattern_opacity"], vals["pattern_spacing"],
                # Header
                vals["head_txt"], vals["head_sz"], vals["head_col"],
                vals["head_y"], vals["head_align"], vals["head_font"],
                # Hero 1&2
                vals["hero1_txt"], vals["hero1_sz"], vals["hero1_col"], vals["hero1_y"],
                vals["hero2_txt"], vals["hero2_sz"], vals["hero2_col"], vals["hero2_y"],
                # Hero 3
                vals["show_hero3"], vals["hero3_txt"], vals["hero3_sz"],
                vals["hero3_col"], vals["hero3_y"],
                # Hero misc
                vals["hero_shadow"], vals["hero_align"], vals["hero_font"],
                # Module
                vals["show_mod"], vals["mod_y"], vals["mod_h"], vals["mod_bg"],
                vals["mod_rad"], vals["show_mod_border"], vals["mod_border_col"],
                vals["mod_shadow"],
                # Badge
                vals["bdg_num"], vals["bdg_title"], vals["bdg_sz"],
                vals["bdg_col"], vals["bdg_shape"],
                # Content
                vals["err_title"], vals["err_txt"], vals["err_col"],
                vals["opt_title"], vals["opt_txt"], vals["opt_col"],
                vals["body_font"],
                # Stats
                vals["show_stats"], vals["stats_y"],
                vals["stats_num1"], vals["stats_lbl1"],
                vals["stats_num2"], vals["stats_lbl2"],
                vals["stats_num3"], vals["stats_lbl3"],
                vals["stats_col"],
                # Logo
                logo_pil, vals["logo_size"], vals["logo_pos"],
                vals["logo_opacity"], vals["show_logo"],
                # Footer
                vals["show_footer"], vals["foot_txt"], vals["foot_col"], vals["foot_sz"],
                # Layout
                vals["margin_x"], vals["spacing"],
                vals["show_divider"], vals["divider_col"], vals["divider_style"],
            )
            psize  = w_preview_size.value
            aspect = img.height / img.width
            thumb  = img.resize((psize, int(psize*aspect)), Image.LANCZOS)
            display(thumb)
            display(HTML(
                f"<div style='color:#607D8B;font-size:12px;margin-top:4px;'>"
                f"⏱️ {ms:.0f}ms &nbsp;|&nbsp; 📐 {img.width}×{img.height}px</div>"
            ))
            current_image = img
    finally:
        _is_rendering = False

def _on_change(change):
    _do_render(_collect_values())

def _on_preview_size_change(change):
    global current_image
    if current_image is None:
        return
    with preview_output:
        clear_output(wait=True)
        psize  = w_preview_size.value
        aspect = current_image.height / current_image.width
        thumb  = current_image.resize((psize, int(psize*aspect)), Image.LANCZOS)
        display(thumb)
        display(HTML(
            f"<div style='color:#607D8B;font-size:12px;'>"
            f"📐 {current_image.width}×{current_image.height}px &nbsp;|&nbsp;"
            f" 🔍 {thumb.width}×{thumb.height}px</div>"
        ))

# Observe all design widgets
for wgt in widget_mapping.values():
    wgt.observe(_on_change, names="value")

# Observe upload widgets separately
w_bg_img_upload.observe(lambda c: _do_render(_collect_values()) if c["name"]=="value" else None, names="value")
w_logo_upload.observe(lambda c: _do_render(_collect_values()) if c["name"]=="value" else None, names="value")

w_preview_size.observe(_on_preview_size_change, names="value")

# ── Undo helpers ──────────────────────────────────────────────
def _save_snapshot():
    _undo_stack.append({k: w.value for k, w in widget_mapping.items()})

def _restore_snapshot():
    if not _undo_stack:
        return False
    state = _undo_stack.pop()
    for k, v in state.items():
        w = widget_mapping.get(k)
        if w is not None:
            try: w.value = v
            except Exception: pass
    return True

# ── JSON Save/Load helpers ────────────────────────────────────
def _save_json():
    data = _collect_values()
    serializable = {}
    for k, v in data.items():
        try:
            _json.dumps(v)
            serializable[k] = v
        except Exception:
            serializable[k] = str(v)
    fname = "template_settings.json"
    with open(fname, "w", encoding="utf-8") as f:
        _json.dump(serializable, f, ensure_ascii=False, indent=2)
    return fname

def _load_json_from_bytes(content_bytes):
    data = _json.loads(content_bytes.decode("utf-8"))
    for k, v in data.items():
        w = widget_mapping.get(k)
        if w is None:
            continue
        try:
            cur = w.value
            if isinstance(cur, bool):   w.value = bool(v)
            elif isinstance(cur, int):  w.value = int(v)
            elif isinstance(cur, float):w.value = float(v)
            else:                       w.value = str(v)
        except Exception:
            pass

print("✅ Live Engine + Undo + Save/Load Ready!")


In [ ]:
# @title 📥 Step 10: Download, Undo & Settings

from google.colab import files

DEFAULTS = {
    "cv_w":1080,"cv_h":1350,"bg_start":"#050814","bg_end":"#1A1030","grad_dir":"vertical",
    "show_accent_bar":True,"accent_bar_h":8,"accent_bar_col":"#00E5FF",
    "bg_img_opacity":0.4,
    "show_pattern":False,"pattern_type":"dots","pattern_col":"#FFFFFF",
    "pattern_opacity":0.08,"pattern_spacing":40,
    "head_txt":"SYSTEM PROTOCOL // ANALYSIS","head_sz":20,"head_col":"#00E5FF",
    "head_y":60,"head_align":"left","head_font":"Regular",
    "hero1_txt":"কেন ছাত্রছাত্রীরা ভুল","hero1_sz":90,"hero1_col":"#FFFFFF","hero1_y":150,
    "hero2_txt":"পদ্ধতিতে পড়াশোনা করে?","hero2_sz":90,"hero2_col":"#00E5FF","hero2_y":270,
    "show_hero3":False,"hero3_txt":"","hero3_sz":70,"hero3_col":"#B0BEC5","hero3_y":390,
    "hero_shadow":True,"hero_align":"left","hero_font":"Bold",
    "show_mod":True,"mod_y":450,"mod_h":420,"mod_bg":"#121826","mod_rad":30,
    "show_mod_border":True,"mod_border_col":"#00E5FF","mod_shadow":True,
    "bdg_num":"01","bdg_title":"রিটেনশন ফ্যাক্টর","bdg_sz":24,"bdg_col":"#00E5FF","bdg_shape":"circle",
    "err_title":"DETECTED ERROR:","err_txt":"মনে মনে পড়া বা প্যাসিভ লার্নিং।","err_col":"#FF5722",
    "opt_title":"OPTIMAL PATH:","opt_txt":"অ্যাক্টিভ রিকল এবং স্পেসড রিপিটেশন।","opt_col":"#00C853",
    "body_font":"Medium",
    "show_stats":False,"stats_y":1050,"stats_col":"#00E5FF",
    "stats_num1":"৯৫%","stats_lbl1":"সাফল্যের হার",
    "stats_num2":"৪৮H","stats_lbl2":"কোর্স সময়",
    "stats_num3":"১০K+","stats_lbl3":"শিক্ষার্থী",
    "show_logo":False,"logo_size":100,"logo_pos":"top-right","logo_opacity":1.0,
    "show_footer":True,"foot_txt":"© Your Name | your.website.com",
    "foot_col":"#607D8B","foot_sz":22,
    "margin_x":80,"spacing":45,
    "show_divider":True,"divider_col":"#00E5FF","divider_style":"solid",
    "export_fmt":"PNG","export_quality":92,"filename":"my_template_design",
}

out_log = widgets.Output()

# Buttons
btn_download = widgets.Button(description="📥 Download Image",  button_style="success",
                               layout=widgets.Layout(width="220px",height="48px"))
btn_reset    = widgets.Button(description="🔄 Reset Defaults",  button_style="warning",
                               layout=widgets.Layout(width="180px",height="48px"))
btn_snapshot = widgets.Button(description="📷 Snapshot (Undo)", button_style="info",
                               layout=widgets.Layout(width="200px",height="48px"))
btn_undo     = widgets.Button(description="↩️ Undo",            button_style="danger",
                               layout=widgets.Layout(width="120px",height="48px"))
btn_save_json= widgets.Button(description="💾 Save Settings",   button_style="primary",
                               layout=widgets.Layout(width="180px",height="48px"))
w_json_upload= widgets.FileUpload(accept=".json", multiple=False,
                                   description="📂 Load Settings:",
                                   layout=widgets.Layout(width="220px"))

def on_download(b):
    with out_log:
        clear_output()
        if current_image is None:
            print("❌ Preview নেই — কোনো slider পরিবর্তন করুন।"); return
        fmt  = w_export_fmt.value
        q    = w_export_quality.value
        name = (w_filename.value.strip() or "template") + "." + fmt.lower().replace("jpeg","jpg")
        kw   = {"format": fmt}
        if fmt in ("JPEG","WEBP"): kw["quality"] = q
        try:
            current_image.save(name, **kw)
            files.download(name)
            print(f"✅ {name} ডাউনলোড হচ্ছে!")
        except Exception as e:
            print(f"❌ {e}")

def on_reset(b):
    _save_snapshot()
    for k, v in DEFAULTS.items():
        w = widget_mapping.get(k)
        if w:
            try: w.value = v
            except Exception: pass
    with out_log:
        clear_output()
        print("✅ Defaults restored! (আগের state undo-তে আছে)")

def on_snapshot(b):
    _save_snapshot()
    with out_log:
        clear_output()
        print(f"📷 Snapshot saved! (stack: {len(_undo_stack)}/5)")

def on_undo(b):
    if _restore_snapshot():
        with out_log:
            clear_output()
            print(f"↩️ Undo done! (remaining: {len(_undo_stack)})")
    else:
        with out_log:
            clear_output()
            print("⚠️ Undo stack খালি।")

def on_save_json(b):
    try:
        fname = _save_json()
        files.download(fname)
        with out_log:
            clear_output()
            print(f"✅ {fname} saved & downloading!")
    except Exception as e:
        with out_log:
            clear_output()
            print(f"❌ {e}")

def on_json_upload(change):
    if not w_json_upload.value:
        return
    try:
        val = w_json_upload.value
        content = list(val.values())[0]["content"] if isinstance(val,dict) else val[0]["content"]
        _save_snapshot()
        _load_json_from_bytes(content)
        with out_log:
            clear_output()
            print("✅ Settings loaded! (আগের state undo-তে আছে)")
    except Exception as e:
        with out_log:
            clear_output()
            print(f"❌ Load error: {e}")

btn_download.on_click(on_download)
btn_reset.on_click(on_reset)
btn_snapshot.on_click(on_snapshot)
btn_undo.on_click(on_undo)
btn_save_json.on_click(on_save_json)
w_json_upload.observe(on_json_upload, names="value")

print("✅ Download + Undo + Save/Load Ready!")


In [ ]:
# @title 🚀 Step 11: Launch Studio

display(HTML("""
<div style="background:linear-gradient(135deg,#1A1030 0%,#050814 100%);
            padding:20px;border-radius:15px;text-align:center;
            border:1px solid #00E5FF;margin-bottom:16px;">
  <h1 style="color:#00E5FF;font-family:sans-serif;margin:0;">
    🎨 Professional Template Studio
    <span style="font-size:13px;color:#69F0AE;">v3 — Full Featured</span>
  </h1>
  <p style="color:#B0BEC5;margin:8px 0 0;">
    Aspect Ratio Presets &nbsp;•&nbsp; Pattern Overlay &nbsp;•&nbsp;
    Top Accent Bar &nbsp;•&nbsp; Hero Line 3 &nbsp;•&nbsp;
    Font Selector &nbsp;•&nbsp; Stats Block &nbsp;•&nbsp;
    BG Image &nbsp;•&nbsp; Logo Upload &nbsp;•&nbsp;
    Save/Load JSON &nbsp;•&nbsp; Undo &nbsp;•&nbsp;
    Badge Shape &nbsp;•&nbsp; Divider Style &nbsp;•&nbsp; Drop Shadow
  </p>
</div>
"""))

# Presets
display(preset_box)

# Tabs
display(ui_tabs)

# Action bar
display(widgets.HTML("<hr>"))
display(widgets.VBox([
    widgets.HBox([btn_download, btn_snapshot, btn_undo, btn_reset, btn_save_json],
                  layout=widgets.Layout(justify_content="center", gap="8px")),
    widgets.HBox([w_json_upload],
                  layout=widgets.Layout(justify_content="center", margin="8px 0 0 0")),
    out_log,
], layout=widgets.Layout(align_items="center")))

# Preview controls
display(widgets.HTML("<hr><b>🖼️ LIVE PREVIEW</b>"))
display(widgets.HBox([w_preview_size,
    widgets.HTML("<span style='color:#607D8B;font-size:12px;margin-left:10px;'>"
                 "preview সাইজ (output resolution অপরিবর্তিত)</span>")],
    layout=widgets.Layout(align_items="center")))
display(preview_output)

# Initial render
_do_render(_collect_values())
